# SR250 UWB (Ultra-Wide Band) Radar Visualisation and Digital Beamforming
# Dataset Creation

## 1. Import required libraries

In this part we install the required libraries for the notebook.

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from PIL import Image
import matplotlib.patches as patches
from typing import Optional, List

import random

## 2. Decluttering

In [2]:
# Size of the moving windows, necessary also for DBF
W = 20

In [3]:
def decluttering_ma(cirs: np.ndarray, W: int) -> np.ndarray:
    """
    Performs decluttering on the time series of CIRs using a moving average
    background subtraction method, as described in Equation (9) of the thesis.

    The decluttered signal s_hat_n,m[k] is calculated as:
    s_hat_n,m[k] = s_n,m[k] - (1/W) * sum(s_n,i[k] for i from m-W+1 to m)

    Args:
        cirs (np.ndarray): Input radar data matrix of shape (time_frames, range_bins),
                           containing complex data.
        W (int): The window dimension (number of subsequent radar frames)
                 to be considered for the moving average. W must be >= 1.

    Returns:
        np.ndarray: The decluttered radar data matrix of the same shape as cirs,
                    containing complex data. The first (W-1) frames will be zeros.
    """
    if cirs.shape[0] == 0:
        return cirs
    
    if not isinstance(W, int) or W < 1:
        raise ValueError("Window size 'W' must be an integer and W >= 1.")

    num_frames, num_range_bins = cirs.shape
    result = np.zeros_like(cirs, dtype=np.complex64)

    # The thesis states: "the first scan of each window must be rejected"
    # This means for frames where a full 'W' window is not available,
    # the output is effectively zero or undefined. We'll set them to zero.
    
    # Loop from the (W-1)-th frame (0-indexed) onwards,
    # as this is the first frame for which a full 'W'-sized window is available.
    for m in range(W - 1, num_frames):
        # Define the window: from (m - W + 1) up to and including m
        # np.mean calculates the mean along axis=0 (frames) for each range bin
        window_average = np.mean(cirs[m - W + 1 : m + 1, :], axis=0)
        
        # Apply the subtraction formula: current_frame - window_average
        result[m, :] = cirs[m, :] - window_average
        
    return result

## 3. Process a sample file

First we load a single .npy file, applying decluttering, then show the time-distance plot obtained as an average of the data coming from the three files.

In [4]:
def load_declutter_sr250_file(filename):
    """
    Loads and declutters a single SR250 antenna file.
    """
    if not os.path.exists(filename):
        print(f"Warning: File not found, skipping: {filename}")
        return None

    raw_frames_full = np.load(filename)  # (time, range_bins)
    # Keep only the first 50 bins
    raw_frames = raw_frames_full[:, :50]
    # Apply decluttering to remove the static background.
    decluttered_matrix = decluttering_ma(raw_frames, W)
    
    return decluttered_matrix

In [5]:
def visualise_sr250(base_filename):
    """
    Loads, processes, and visualizes SR250 data from three antennas.
    Returns the list of complex decluttered matrices and their magnitudes.
    """
    # Automatically find paths for all three antennas
    file_list = []
    if "_sr250_rx0" in base_filename:
        base_path = base_filename.replace("_sr250_rx0.npy", "")
        for i in range(3):
            file_list.append(f"{base_path}_sr250_rx{i}.npy")
    else:
        print("Error: Please provide the path to an 'sr250_rx0' file.")
        return [], [] # Return empty lists for both complex and magnitude

    # --- Process each antenna and store the resulting complex matrix ---
    all_decluttered_complex = []
    all_magnitudes = []
    for f in file_list:
        # Process the file to get the complex decluttered matrix
        decluttered_result = load_declutter_sr250_file(f)
        if decluttered_result is not None:
            # Remove the first W time frames from the decluttered data
            decluttered_result = decluttered_result[W:, :]
            # Store complex data
            all_decluttered_complex.append(decluttered_result)
            # Also store magnitude for original visualization if needed
            all_magnitudes.append(np.abs(decluttered_result))
    
    if not all_decluttered_complex:
        print("Error: No data was processed. Exiting.")
        return [], []
    
    # Only complex data is needed for beamforming, so we return just that.
    return all_decluttered_complex

## 4. Apply Digital Beamforming

In [6]:
# --- Radar Parameters ---
d = 1.75e-2          # Distance between rx antenna elements for azimuth
f = 7.987e9          # Center frequency of radar pulse (Hz) -> 7.737 - 8.237 GHz
c = 299792458        # Speed of light (m/s)

l = c/f              # Wavelength
k = 2*np.pi/l        # Wavenumber

# Beam steering angles for azimuth
min_angle = -45
max_angle = 45
num_angles = round((max_angle - min_angle + 1) / 2)
print("Number of angles considered: ", num_angles)
beam_steering_angles = np.deg2rad(np.linspace(min_angle, max_angle, num_angles))

# Receivers to be considered, RX0 and RX2
receivers = [0, 2]
# Initial time frame for window plot
initial_time_frame = 0
# Target time frame for window plot, 20 time frames correspond to 1 second
target_time_frame = 20

Number of angles considered:  46


In [7]:
# --- Beamforming Helper Functions ---
def compute_bf_weights(angle_range, delta, k, num_rx_elements):
    """
    Computes beamforming weights for a linear array.
    
    Args:
        angle_range (np.ndarray): Array of angles (in radians) for which to compute weights.
        delta (float): Distance between adjacent antenna elements.
        k (float): Wavenumber (2*pi/lambda).
        num_rx_elements (int): Number of receive antenna elements.
        
    Returns:
        np.ndarray: Beamforming weights of shape (len(angle_range), num_rx_elements).
    """
    bf_weights = np.zeros((angle_range.shape[0], num_rx_elements), dtype="complex")
    for i in range(angle_range.shape[0]):      # Loop over angles
        for j in range(num_rx_elements):       # Loop over antenna elements
            # Phase shift for element j at angle_range[i]
            bf_weights[i, j] = np.exp(-1j * k * delta * j * np.sin(angle_range[i]))
    return bf_weights

In [8]:
def perform_azimuth_beamforming(rx0_complex_data, rx1_complex_data, d, k, beam_steering_angles):
    """
    Performs azimuth beamforming on complex data from rx0 and rx1 antennas.
    This version processes all time frames individually, producing a 3D output.
    
    Args:
        rx0_complex_data (np.ndarray): Complex CIRs from rx0 (time_frames, range_bins).
        rx1_complex_data (np.ndarray): Complex CIRs from rx1 (time_frames, range_bins).
        d (float): Distance between rx0 and rx1 antennas (element spacing).
        k (float): Wavenumber (2*pi/lambda).
        beam_steering_angles (np.ndarray): Array of angles in radians for beamsteering.

    Returns:
        np.ndarray: Beamformed data of shape (time_frames, range_bins, angles), or None if error.
    """

    # Resize the range bins due to slight mismatch
    rx0_complex_data = rx0_complex_data[:, 0:45]
    rx1_complex_data = rx1_complex_data[:, 2:47]
    
    if rx0_complex_data is None or rx1_complex_data is None:
        print("Error: Input data for beamforming is missing.")
        return None
    if rx0_complex_data.shape != rx1_complex_data.shape:
        print("Error: rx0 and rx1 data shapes do not match. Cannot perform beamforming.")
        return None
    else:
        print(f"Shape of the antennas: {rx0_complex_data.shape}, with dtype {rx0_complex_data.dtype}")
    
    num_time_frames = rx0_complex_data.shape[0]   # Slow time
    num_range_bins = rx0_complex_data.shape[1]    # Fast time
    num_rx_elements = 2
    num_angles = beam_steering_angles.shape[0]    # Number of angles to be considered

    # Compute beamforming weights. These are constant for all time frames and range bins.
    bf_weights = compute_bf_weights(beam_steering_angles, d, k, num_rx_elements)

    # Initialize a 3D array to store beamformed results
    beamformed_3d_result = np.zeros((num_time_frames, num_range_bins, num_angles), dtype=complex)

    # This version performs the same carried out by bf.py but exploiting NumPY vectorized operations.
    for t in range(num_time_frames):
        # Extract the complex CIRs for the current time frame from both RX antennas       
        # and stack them to form the input for beamforming for the current frame.
        # data_for_current_frame_all_bins will have shape (num_rx_elements, range_bins)
        data_for_current_frame_all_bins = np.stack([rx0_complex_data[t, :], rx1_complex_data[t, :]], axis=0)

        # Perform matrix multiplication for all range bins in the current frame simultaneously
        # (num_angles, num_rx_elements) @ (num_rx_elements, num_range_bins)
        # Result of dot product: (num_angles, num_range_bins)
        beamformed_slice_transposed = np.dot(bf_weights, data_for_current_frame_all_bins)
        
        # Transpose the result to get (range_bins, angles) for this time slice
        beamformed_3d_result[t, :, :] = beamformed_slice_transposed.T
            
    print("Beamforming complete for all frames and range bins.")
    return beamformed_3d_result

In [9]:
def show_range_angle_plot(matrix, angles_rad, title="Beamformed Range-Azimuth Image", 
                          range_conversion_factor=10, output_dir= "/kaggle/working/", # Changed parameter name
                          gt_highlight_coords_list: Optional[List[tuple]] = None):
    """
    Visualizes the beamformed data as a 2D heatmap (Distance vs. Azimuth Angle).
    Converts range bins to meters for visualization.
    Optionally draws cross markers at specified ground truth positions.
    Image is saved to output_dir with the given title.
    """
    # Construct the full output path for the image file
    image_filename = f"{title}.png"
    full_output_path = os.path.join(output_dir, image_filename) # Correctly join directory and filename
    os.makedirs(output_dir, exist_ok=True) # Ensure the directory exists

    OUTPUT_IMAGE_SIZE = (160, 160)
    dpi = 100
    
    print("Generating Beamformed plot...\n")
    heatmap_data = matrix
    fig, ax = plt.subplots(figsize=(12, 12)) 

    angles_deg = np.rad2deg(angles_rad)

    min_range_meters = 0
    max_range_meters = heatmap_data.shape[0] / range_conversion_factor 

    extent = [angles_deg.min(), angles_deg.max(), min_range_meters, max_range_meters] 

    im = ax.imshow(
        heatmap_data, 
        aspect='auto', 
        origin='lower', 
        cmap='gist_gray',
        interpolation='nearest',
        vmin=np.min(heatmap_data),
        vmax=np.max(heatmap_data),
        extent=extent 
    )
    
    ax.axis('off')
    try:
        fig.savefig(full_output_path, bbox_inches='tight', pad_inches=0, dpi=dpi)
        print(f"Image saved successfully to: {full_output_path}") # Confirmation print
    except Exception as e:
        print(f"Error saving image to {full_output_path}: {e}") # Error handling for save
    
    ax.axis('on')
    ax.set_title(title)
    ax.set_xlabel("Azimuth Angle (degrees)")
    ax.set_ylabel("Distance (meters)")
    fig.colorbar(im, ax=ax, label="Beamformed Signal Magnitude")
    
    num_ticks_x = 7 
    tick_indices_x = np.linspace(0, len(angles_deg) - 1, num_ticks_x).astype(int)
    ax.set_xticks(angles_deg[tick_indices_x])
    ax.set_xticklabels([f"{a:.0f}°" for a in angles_deg[tick_indices_x]])

    if max_range_meters <= 1:
        y_tick_interval = 0.1
    elif max_range_meters <= 5:
        y_tick_interval = 0.5
    else:
        y_tick_interval = 1.0
    
    y_ticks = np.arange(min_range_meters, max_range_meters + y_tick_interval, y_tick_interval)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([f"{d:.1f}m" for d in y_ticks])
    
    plt.tight_layout()
    # plt.show() # Keep commented out for batch processing
    plt.close(fig) # Close the figure to free up memory

In [10]:
def normalize_range_angle_map(matrix: np.ndarray) -> np.ndarray:
    """
    Normalizes a 2D Range-Angle map to the range [0, 1] using min-max scaling,
    as described in Equation (11) of the thesis.

    Args:
        matrix (np.ndarray): The input 2D Range-Angle map.

    Returns:
        np.ndarray: The normalized map.
    """
    min_val = np.min(matrix)
    max_val = np.max(matrix)
    if max_val == min_val: # Avoid division by zero for uniform matrices
        return np.zeros_like(matrix)
    normalized_matrix = (matrix - min_val) / (max_val - min_val)
    return normalized_matrix

In [11]:
def apply_beamforming (base_file, rx_s = [0, 2], avg_mode='coherent', one=False, init_tf=0, target_tf=20):
    # Load and declutter data for all antennas
    print(f"\n--- Loading and processing data for beamforming: {os.path.basename(base_file)} ---")
    all_complex_data = visualise_sr250(base_file)

    # Check if data are available for azimuth beamforming.
    antenna1_data = all_complex_data[rx_s[0]]
    antenna2_data = all_complex_data[rx_s[1]]

    # Perform azimuth beamforming, return a 3D array (time_frames, range_bins, angles)
    beamformed_3d_data = perform_azimuth_beamforming(antenna1_data, antenna2_data, d, k, beam_steering_angles)
    
    if beamformed_3d_data is None:
        print("Beamforming failed. Exiting apply_beamforming.")
        return None # Indicate failure

    print(f"Shape of the full beamformed data: {beamformed_3d_data.shape} with dtype {beamformed_3d_data.dtype}")
    
    # Return the full 3D beamformed data
    return beamformed_3d_data

In [12]:
def save_and_show_file (bf_data, base_filename, init_tf=0, target_tf=20): # Removed 'show' parameter
    """
    Aggregates beamformed data for a specific time window and saves the resulting
    range-azimuth heatmap image.
    
    Args:
        bf_data (np.ndarray): The full 3D beamformed data (time_frames, range_bins, angles).
        base_filename (str): The original .npy file path (used for deriving image title).
        init_tf (int): The starting time frame for the window.
        target_tf (int): The ending time frame for the window (exclusive).
    
    Returns:
        str: The generated title of the image (e.g., "filename_part__start-end").
    """
    initial_time_frame = max(0, init_tf)
    target_time_frame = min(target_tf, bf_data.shape[0])
    
    # Apply coherent averaging over the specified time window
    aggregated_beamformed_data = np.abs(np.sum(bf_data[initial_time_frame:target_time_frame, :, ], axis=0))
    
    # Normalization for visualization
    normalized_beamformed_data = normalize_range_angle_map(aggregated_beamformed_data)
    
    # Construct the title/filename base from the original filename, stripping the radar-specific suffix
    base_file_part = os.path.basename(base_filename).replace('_sr250_rx0.npy', '')
    title = f"{base_file_part}__{initial_time_frame}-{target_time_frame}"
    
    # Always call show_range_angle_plot to ensure the image is saved.
    # The output_dir is passed explicitly.
    show_range_angle_plot(normalized_beamformed_data, beam_steering_angles, title, output_dir="/kaggle/working/")

    return title

## Define the dataset

In [13]:
def extract_scenario_numbers(filepath: str) -> List[str]:
    """
    Extracts a list of scenario numbers from a SR250 radar file path.
    These numbers are typically hyphen-separated digits found immediately before
    '_Still position_' or '_Moving position_'.
    This enhanced version handles cases with trailing hyphens, multiple hyphens
    between numbers, and optional whitespace before the '_Still position_' or
    '_Moving position_' marker.

    Examples based on common interpretations of such filenames:
    - "/path/to/12-13-14_Still position_...rx0.npy" -> ['12', '13', '14']
    - "/path/to/5_Moving position_...rx0.npy" -> ['5']
    - "/path/to/(move)2-12-9-3-13-14-4-8-13-2_Still position_...rx0.npy" -> ['2', '12', '9', '3', '13', '14', '4', '8', '13', '2']
    - "/path/to/7-8-9-_Still position_...rx0.npy" -> ['7', '8', '9']
    - "/path/to/2--8-14_Still position_...rx0.npy" -> ['2', '8', '14']
    - "/path/to/2-12-14 _Still position_20250618-115119_sr250_rx0.npy" -> ['2', '12', '14'] (NEW case handled)
    - "/path/to/test-no-numbers_Still position_...rx0.npy" -> [] (no numbers in the specific pattern)
    - "/path/to/no_position_marker_file.npy" -> [] (no position marker)

    Args:
        filepath (str): The full path to the SR250 radar data file.

    Returns:
        List[str]: A list of strings, where each string is an extracted number.
                   Returns an empty list if no such pattern is found.
    """
    filename = os.path.basename(filepath)

    # '([0-9-]+)': This captures one or more occurrences of digits (0-9) or hyphens (-).
    # This will capture strings like "7-8-9-" or "2--8-14".
    # The subsequent split and filter will clean these into actual numbers.
    # '\s*': Matches zero or more whitespace characters. This is the key addition.
    pattern = r'([0-9-]+)\s*_(?:Still|Moving) position_' # MODIFIED LINE

    match = re.search(pattern, filename)

    if match:
        numbers_str = match.group(1) # Get the captured sequence (e.g., "7-8-9-", "2--8-14")
        # Split the captured string by '-' and then use a list comprehension
        # to filter out any empty strings that result from multiple or trailing hyphens.
        return [num for num in numbers_str.split('-') if num]
    else:
        return []

In [14]:
# Width and height are referred to plot a figure of size (12, 12)
IMAGE_WIDTH = 930
IMAGE_HEIGHT = 924

# Bounding boxes size
BB_SIZE = 100  # (BB_SIZE x BB_SIZE)

# Min and max values for axis
MIN_DISTANCE = 0
MAX_DISTANCE = 4.5

# Values for the distance
high_dist = round(((IMAGE_HEIGHT / MAX_DISTANCE) * 1) - BB_SIZE / 2, 1)
med_dist = round(((IMAGE_HEIGHT / MAX_DISTANCE) * 2) - BB_SIZE / 2, 1)
low_dist = round(((IMAGE_HEIGHT / MAX_DISTANCE) * 3) - BB_SIZE / 2, 1)

# Values for the angle
left_angle = round((IMAGE_WIDTH * 0.25) - BB_SIZE / 2, 1)
med_angle = round((IMAGE_WIDTH * 0.5) - BB_SIZE / 2, 1)
right_angle = round((IMAGE_WIDTH * 0.75) - BB_SIZE / 2, 1)

In [15]:
# Define the map (position) -> (x, y)
positions_map = {
    "2": (right_angle, low_dist),
    "3": (med_angle, low_dist),
    "4": (left_angle, low_dist),
    "7": (right_angle, med_dist),
    "8": (med_angle, med_dist),
    "9": (left_angle, med_dist),
    "12": (right_angle, high_dist),
    "13": (med_angle, high_dist),
    "14": (left_angle, high_dist)
}
print(f"Defined the following positions on an image of size ({IMAGE_WIDTH}, {IMAGE_HEIGHT}): \n{positions_map}")

Defined the following positions on an image of size (930, 924): 
{'2': (647.5, 566.0), '3': (415.0, 566.0), '4': (182.5, 566.0), '7': (647.5, 360.7), '8': (415.0, 360.7), '9': (182.5, 360.7), '12': (647.5, 155.3), '13': (415.0, 155.3), '14': (182.5, 155.3)}


In [16]:
import json
import os

# Define the basic structure of a .labels file.
# This ensures consistency if a new file is being created from scratch.
DEFAULT_LABELS_STRUCTURE = {
    "version": 1,
    "type": "bounding-box-labels",
    "boundingBoxes": {}
}

## Process a single image

In [17]:
# Define constants for the sliding window
WINDOW_SIZE = 20 # The size of each time window for beamforming aggregation
STEP_SIZE = 50   # The step size for sliding the window

In [18]:
def process_all_samples(dataset_base_path: str = "/kaggle/input/haeeai-project-uwb/SR250Mate/", 
                        output_labels_filename: str = "bounding_boxes.labels"):
    """
    Processes all SR250 radar data files ending with '_sr250_rx0.npy' in the given dataset path,
    generates beamformed images for sliding time windows, and creates a single consolidated
    JSON labels file with bounding box annotations for all generated images.

    Args:
        dataset_base_path (str): The base directory containing the SR250 radar .npy files.
        output_labels_filename (str): The name of the consolidated .labels JSON file to be created.
    """
    print(f"--- Starting processing of all samples in: {dataset_base_path} ---")

    consolidated_labels_data = DEFAULT_LABELS_STRUCTURE.copy()
    all_bboxes_for_json = {} 
    
    output_dir_for_images_and_labels = "/kaggle/working/"
    os.makedirs(output_dir_for_images_and_labels, exist_ok=True)

    found_files = []
    for root, _, files in os.walk(dataset_base_path):
        for file in files:
            # Look for files ending with '_sr250_rx0.npy'
            if file.endswith("_sr250_rx0.npy"):  
                full_filepath = os.path.join(root, file)
                found_files.append(full_filepath)
    
    if not found_files:
        print(f"No '_sr250_rx0.npy' files found in {dataset_base_path}. Exiting.")
        return

    print(f"Found {len(found_files)} radar files to process.")

    for i, base_filename in enumerate(found_files):
        print(f"\n\nProcessing file {i+1}/{len(found_files)}: {os.path.basename(base_filename)}")
        
        # 1. Extract labels from the filename (these labels are static for all windows of this file)
        labels = extract_scenario_numbers(base_filename)
        print("Extracted labels for this file: ", labels)

        # 2. Load the files, declutter and apply beamforming once for the entire data
        # bf_data will be (num_total_frames, range_bins, angles)
        bf_data = apply_beamforming(base_filename)

        if bf_data is None:
            print(f"Skipping processing of {os.path.basename(base_filename)} due to beamforming failure.")
            continue 
        
        num_total_frames = bf_data.shape[0]
        print(f"Total time frames available for {os.path.basename(base_filename)}: {num_total_frames}")

        # 3. Create the common bounding box list for this original file
        common_bboxes_for_this_file = []
        for label_str in labels:
            if not label_str: # Skip empty strings from extraction (e.g., "2--8-14")
                continue
                
            if label_str in positions_map:
                x_coord, y_coord = positions_map[label_str]
                bbox = {
                    "label": label_str,
                    "x": float(x_coord),
                    "y": float(y_coord),
                    "width": float(BB_SIZE),
                    "height": float(BB_SIZE)
                }
                common_bboxes_for_this_file.append(bbox)
            else:
                print(f"Warning: Label '{label_str}' extracted from '{os.path.basename(base_filename)}' "
                      f"not found in positions_map. Skipping bounding box for it.")

        # 4. Iterate through sliding windows and generate images and JSON entries
        # The loop iterates from 0 up to the last possible start frame that allows a full WINDOW_SIZE
        for start_tf in range(0, num_total_frames - WINDOW_SIZE + 1, STEP_SIZE):
            end_tf = start_tf + WINDOW_SIZE
            
            print(f"  Processing window {start_tf}-{end_tf}...")

            # Call save_and_show_file to generate and save the image for this window
            # The 'show' parameter is no longer needed/passed
            generated_image_full_title = save_and_show_file(bf_data, base_filename, init_tf=start_tf, target_tf=end_tf)
            
            # The key for the bounding box in the JSON is the full image filename (e.g., "filename_part__0-20.png")
            image_filename_for_json_key = generated_image_full_title + ".png"
            
            # Add the bounding boxes for this specific window's image to the consolidated dictionary.
            # All windows from the same original file share the same labels/bboxes.
            all_bboxes_for_json[image_filename_for_json_key] = common_bboxes_for_this_file

    # 5. Populate the consolidated JSON structure and write it to a single file
    consolidated_labels_data["boundingBoxes"] = all_bboxes_for_json
    
    final_labels_filepath = os.path.join(output_dir_for_images_and_labels, output_labels_filename)

    try:
        with open(final_labels_filepath, 'w') as f:
            json.dump(consolidated_labels_data, f, indent=2) 
        print(f"\n--- Successfully wrote all consolidated bounding box labels to: {final_labels_filepath} ---")
    except Exception as e:
        print(f"\n--- Error writing consolidated JSON file {final_labels_filepath}: {e} ---")

In [19]:
process_all_samples(dataset_base_path="/kaggle/input/haeeai-project-uwb/SR250Mate/",
                    output_labels_filename="bounding_boxes.labels")

--- Starting processing of all samples in: /kaggle/input/haeeai-project-uwb/SR250Mate/ ---
Found 90 radar files to process.


Processing file 1/90: 13-14_Still position_20250618-141413_sr250_rx0.npy
Extracted labels for this file:  ['13', '14']

--- Loading and processing data for beamforming: 13-14_Still position_20250618-141413_sr250_rx0.npy ---
Shape of the antennas: (1200, 45), with dtype complex64
Beamforming complete for all frames and range bins.
Shape of the full beamformed data: (1200, 45, 46) with dtype complex128
Total time frames available for 13-14_Still position_20250618-141413_sr250_rx0.npy: 1200
  Processing window 0-20...
Generating Beamformed plot...

Image saved successfully to: /kaggle/working/13-14_Still position_20250618-141413__0-20.png
  Processing window 50-70...
Generating Beamformed plot...

Image saved successfully to: /kaggle/working/13-14_Still position_20250618-141413__50-70.png
  Processing window 100-120...
Generating Beamformed plot...

Image saved suc